# 📘 Lakehouse Architecture Patterns
## Lambda, Kappa, Medallion, Data Mesh & Hybrid Approaches

**What you'll learn:**
- The evolution of data architecture (warehouse → lake → lakehouse)
- Lambda Architecture (batch + speed layers)
- Kappa Architecture (streaming-only)
- Medallion Architecture (Bronze/Silver/Gold)
- Data Mesh (domain-oriented decentralization)
- Hybrid patterns and when to combine approaches
- Architecture decision framework
- Real-world case studies and trade-offs
- How to choose the right architecture for your organization

**Prerequisites:** Notebooks 00A (DE Complete Guide), 04 (Medallion), 24 (Streaming)
**Estimated Time:** 5-6 hours

---

---
# 📗 Section 1: The Evolution of Data Architecture

## From Warehouses to Lakehouses

```
1990s: DATA WAREHOUSE ERA
├── Structured data only (SQL)
├── Expensive proprietary systems (Teradata, Oracle)
├── Schema-on-write (rigid)
├── Great for BI, terrible for ML
└── Example: Walmart's 40PB Teradata warehouse

2010s: DATA LAKE ERA
├── Store everything (structured + unstructured)
├── Cheap object storage (S3, HDFS)
├── Schema-on-read (flexible)
├── Great for ML, terrible for BI (data swamp)
└── Example: Netflix's 100PB S3 data lake

2015+: TWO-TIER ARCHITECTURE (Lake + Warehouse)
├── Lake for raw/ML, Warehouse for BI
├── Data copied between systems (expensive, stale)
├── Two systems to manage (complex)
└── Example: Most enterprises pre-2020

2020+: LAKEHOUSE ERA
├── One system for everything (Lake + Warehouse features)
├── ACID transactions on object storage
├── Schema enforcement + flexibility
├── Great for BOTH BI and ML
└── Example: Databricks, Delta Lake, Iceberg
```

## Why Lakehouses Won

| Problem | Warehouse | Lake | Lakehouse |
|---------|-----------|------|-----------|
| Store any data type | ❌ Structured only | ✅ Anything | ✅ Anything |
| ACID transactions | ✅ | ❌ | ✅ |
| Schema enforcement | ✅ | ❌ | ✅ (configurable) |
| Time travel | Limited | ❌ | ✅ |
| Cost | $$$ | $ | $ |
| BI performance | ✅ | ❌ | ✅ |
| ML support | ❌ | ✅ | ✅ |
| Open formats | ❌ (proprietary) | ✅ | ✅ (Parquet/Delta) |

In [0]:
# Architecture evolution timeline
architectures = {
    "Data Warehouse (1990s-present)": {
        "philosophy": "Structured, schema-on-write, optimized for SQL",
        "strengths": ["Fast SQL queries", "ACID transactions", "Strong governance"],
        "weaknesses": ["Expensive", "Structured only", "Vendor lock-in", "Poor ML support"],
        "examples": ["Teradata", "Oracle", "Snowflake", "BigQuery", "Redshift"],
        "best_for": "BI reporting, regulatory compliance, structured analytics"
    },
    "Data Lake (2010s-present)": {
        "philosophy": "Store everything cheaply, process later",
        "strengths": ["Cheap storage", "Any data type", "ML-friendly", "Open formats"],
        "weaknesses": ["No ACID", "Data swamp risk", "Poor BI performance", "No governance"],
        "examples": ["Hadoop/HDFS", "S3 + Hive", "ADLS + Spark"],
        "best_for": "ML training, raw data archive, unstructured data"
    },
    "Lakehouse (2020s-present)": {
        "philosophy": "Best of both: cheap storage + warehouse features",
        "strengths": ["ACID on object storage", "Any data type", "BI + ML", "Open formats", "Time travel"],
        "weaknesses": ["Newer (less mature tooling)", "Requires Delta/Iceberg knowledge"],
        "examples": ["Databricks + Delta Lake", "Apache Iceberg", "Apache Hudi"],
        "best_for": "Everything — unified platform for all data workloads"
    }
}

print("📊 DATA ARCHITECTURE EVOLUTION")
print("=" * 70)
for name, details in architectures.items():
    print(f"\n🏗️ {name}")
    print(f"   Philosophy: {details['philosophy']}")
    print(f"   Strengths: {', '.join(details['strengths'])}")
    print(f"   Weaknesses: {', '.join(details['weaknesses'])}")
    print(f"   Examples: {', '.join(details['examples'])}")
    print(f"   Best for: {details['best_for']}")


---
# 📗 Section 2: Lambda Architecture

## Overview

The Lambda Architecture processes data through TWO parallel paths:
- **Batch layer**: Processes ALL historical data (complete but slow)
- **Speed layer**: Processes only recent data (fast but approximate)
- **Serving layer**: Merges results from both for queries

```
                         ┌─────────────────────────────────────┐
                         │           DATA SOURCES               │
                         └──────────────────┬──────────────────┘
                                            │
                         ┌──────────────────┼──────────────────┐
                         │                  │                  │
                   ┌─────▼─────┐      ┌─────▼─────┐          │
                   │   BATCH   │      │   SPEED   │          │
                   │   LAYER   │      │   LAYER   │          │
                   │           │      │           │          │
                   │ • All data│      │ • Recent  │          │
                   │ • Complete│      │   data    │          │
                   │ • Slow    │      │ • Fast    │          │
                   │ • Spark   │      │ • Flink   │          │
                   │   batch   │      │ • Kafka   │          │
                   └─────┬─────┘      └─────┬─────┘          │
                         │                  │                  │
                   ┌─────▼──────────────────▼─────┐           │
                   │       SERVING LAYER          │           │
                   │                              │           │
                   │  Merge batch + speed results │           │
                   │  Serve to queries/dashboards │           │
                   └──────────────────────────────┘           │
```

## When to Use Lambda

| Scenario | Lambda? | Why |
|----------|---------|-----|
| Need both real-time AND complete historical | ✅ Yes | Only Lambda gives both |
| Real-time fraud detection + daily reports | ✅ Yes | Speed layer for fraud, batch for reports |
| Simple daily ETL pipeline | ❌ No | Overkill — use Medallion |
| Pure streaming use case | ❌ No | Use Kappa instead |

## Pros and Cons

| Pros | Cons |
|------|------|
| Complete + real-time | Two codebases to maintain |
| Fault-tolerant (batch corrects speed errors) | Complex merging logic |
| Proven at scale (LinkedIn, Twitter) | Higher operational cost |
| Each layer can use best-fit technology | Eventual consistency between layers |

## Production Example: LinkedIn Activity Feed

LinkedIn uses Lambda for their activity feed:
- **Batch**: Spark processes all activity data nightly (complete, accurate)
- **Speed**: Kafka Streams processes real-time activity (fast, approximate)
- **Serving**: Merges both for the feed you see (recent = speed, older = batch)

In [0]:
# Lambda Architecture simulation
class LambdaArchitecture:
    """Simulates a Lambda Architecture with batch + speed layers."""
    
    def __init__(self):
        self.batch_results = {}  # Complete but stale
        self.speed_results = {}  # Recent but approximate
        self.raw_data = []
    
    def ingest(self, events):
        """Ingest events into both layers."""
        self.raw_data.extend(events)
        
        # Speed layer processes immediately (approximate)
        for event in events:
            key = event["key"]
            if key not in self.speed_results:
                self.speed_results[key] = {"count": 0, "total": 0}
            self.speed_results[key]["count"] += 1
            self.speed_results[key]["total"] += event["value"]
    
    def run_batch(self):
        """Batch layer recomputes from ALL data (complete, accurate)."""
        self.batch_results = {}
        for event in self.raw_data:
            key = event["key"]
            if key not in self.batch_results:
                self.batch_results[key] = {"count": 0, "total": 0, "avg": 0}
            self.batch_results[key]["count"] += 1
            self.batch_results[key]["total"] += event["value"]
            self.batch_results[key]["avg"] = (
                self.batch_results[key]["total"] / self.batch_results[key]["count"]
            )
        # After batch completes, clear speed layer (batch is now authoritative)
        self.speed_results = {}
    
    def query(self, key):
        """Serving layer: merge batch + speed results."""
        batch = self.batch_results.get(key, {"count": 0, "total": 0})
        speed = self.speed_results.get(key, {"count": 0, "total": 0})
        
        # Merge: batch (historical) + speed (recent)
        return {
            "count": batch["count"] + speed["count"],
            "total": batch["total"] + speed["total"],
            "source": "batch+speed" if speed["count"] > 0 else "batch_only"
        }


# Demo
lambda_arch = LambdaArchitecture()

# Day 1: Historical data (processed by batch)
historical = [
    {"key": "product_A", "value": 100, "timestamp": "2024-03-14"},
    {"key": "product_A", "value": 150, "timestamp": "2024-03-14"},
    {"key": "product_B", "value": 200, "timestamp": "2024-03-14"},
]
lambda_arch.ingest(historical)
lambda_arch.run_batch()  # Batch processes all historical data

# Day 2: Real-time events (speed layer only, batch hasn't run yet)
realtime = [
    {"key": "product_A", "value": 175, "timestamp": "2024-03-15T10:00:00"},
    {"key": "product_C", "value": 50, "timestamp": "2024-03-15T10:01:00"},
]
lambda_arch.ingest(realtime)

print("🏗️ LAMBDA ARCHITECTURE DEMO")
print("=" * 60)
print("\n📊 Query Results (serving layer merges batch + speed):")
for key in ["product_A", "product_B", "product_C"]:
    result = lambda_arch.query(key)
    print(f"   {key}: count={result['count']}, total=${result['total']:.0f} "
          f"(source: {result['source']})")

print("\n💡 Key Insight:")
print("   product_A: 2 from batch (yesterday) + 1 from speed (today) = 3 total")
print("   product_B: 1 from batch only (no new events today)")
print("   product_C: 1 from speed only (new product, batch hasn't seen it yet)")
print("\n   When batch runs tonight, it will recompute ALL data accurately.")
print("   Speed layer results are then cleared (batch is authoritative).")


---
# 📗 Section 3: Kappa Architecture

## Overview

Kappa simplifies Lambda by removing the batch layer entirely.
ALL data is processed as a stream — even historical data.

```
┌──────────────┐    ┌──────────────────┐    ┌──────────────┐    ┌──────────────┐
│    DATA      │───▶│    STREAMING     │───▶│  MATERIALIZED│───▶│    QUERY     │
│   SOURCES    │    │     LAYER        │    │    VIEWS     │    │    LAYER     │
│              │    │                  │    │              │    │              │
│ All data as  │    │ Single codebase  │    │ Pre-computed │    │ BI, APIs,    │
│ events       │    │ (Kafka + Flink)  │    │ results      │    │ dashboards   │
└──────────────┘    └──────────────────┘    └──────────────┘    └──────────────┘
```

## Kappa vs Lambda

| Aspect | Lambda | Kappa |
|--------|--------|-------|
| **Codebases** | Two (batch + streaming) | One (streaming only) |
| **Complexity** | Higher (maintain two systems) | Lower (one system) |
| **Reprocessing** | Batch layer recomputes | Replay from event log |
| **Latency** | Speed layer: real-time | Everything: real-time |
| **Accuracy** | Batch corrects speed errors | Must be correct first time |
| **Cost** | Higher (two systems running) | Lower (one system) |
| **Best for** | Mixed batch + real-time needs | Event-driven, streaming-first |

## When to Use Kappa

| Scenario | Kappa? | Why |
|----------|--------|-----|
| Event-driven microservices | ✅ Yes | Natural fit for event streams |
| Real-time analytics only | ✅ Yes | No batch needed |
| IoT sensor processing | ✅ Yes | Continuous data flow |
| Need to reprocess years of history | ❌ No | Expensive to replay years of events |
| Complex batch transformations | ❌ No | Some logic is easier in batch |

## The Key Requirement: Replayable Event Log

Kappa REQUIRES an immutable, replayable event log (like Kafka with long retention):

```
Kafka Topic (retain forever or very long):
├── Event 1 (2024-01-01)
├── Event 2 (2024-01-01)
├── ...
├── Event 1,000,000 (2024-03-15)
└── Event 1,000,001 (2024-03-15)

Need to reprocess? → Reset consumer offset → Replay from beginning
```

## Production Example: Uber's Real-Time Analytics

Uber uses Kappa for real-time pricing and dispatch:
- **Event log**: Kafka (1 trillion messages/day, retained for 3 days)
- **Processing**: Apache Flink (stateful stream processing)
- **Serving**: Materialized views in Pinot (real-time OLAP)
- **Reprocessing**: Replay from Kafka when logic changes

In [0]:
# Kappa Architecture simulation
from collections import defaultdict

class KappaArchitecture:
    """Simulates a Kappa Architecture (streaming-only)."""
    
    def __init__(self):
        self.event_log = []  # Immutable, append-only (like Kafka)
        self.consumer_offset = 0  # Where we've processed up to
        self.materialized_views = defaultdict(lambda: {"count": 0, "total": 0, "avg": 0})
    
    def produce(self, events):
        """Append events to the immutable log."""
        for event in events:
            event["offset"] = len(self.event_log)
            self.event_log.append(event)
    
    def process(self):
        """Process events from current offset (streaming consumer)."""
        new_events = self.event_log[self.consumer_offset:]
        
        for event in new_events:
            key = event["key"]
            self.materialized_views[key]["count"] += 1
            self.materialized_views[key]["total"] += event["value"]
            self.materialized_views[key]["avg"] = (
                self.materialized_views[key]["total"] / 
                self.materialized_views[key]["count"]
            )
        
        processed = len(new_events)
        self.consumer_offset = len(self.event_log)
        return processed
    
    def reprocess(self):
        """Reprocess ALL events from beginning (Kappa's 'batch' equivalent)."""
        # Reset state
        self.materialized_views = defaultdict(lambda: {"count": 0, "total": 0, "avg": 0})
        self.consumer_offset = 0
        # Replay all events
        return self.process()
    
    def query(self, key):
        """Query materialized view."""
        return dict(self.materialized_views.get(key, {"count": 0, "total": 0, "avg": 0}))


# Demo
kappa = KappaArchitecture()

# Produce events
kappa.produce([
    {"key": "orders", "value": 100, "ts": "2024-03-15T10:00"},
    {"key": "orders", "value": 150, "ts": "2024-03-15T10:01"},
    {"key": "orders", "value": 200, "ts": "2024-03-15T10:02"},
    {"key": "returns", "value": 50, "ts": "2024-03-15T10:03"},
])

# Process (streaming)
processed = kappa.process()

print("🌊 KAPPA ARCHITECTURE DEMO")
print("=" * 60)
print(f"\n   Events in log: {len(kappa.event_log)}")
print(f"   Processed: {processed}")
print(f"\n   Materialized Views:")
print(f"   orders: {kappa.query('orders')}")
print(f"   returns: {kappa.query('returns')}")

# New events arrive
kappa.produce([
    {"key": "orders", "value": 300, "ts": "2024-03-15T10:04"},
])
kappa.process()
print(f"\n   After new event:")
print(f"   orders: {kappa.query('orders')}")

# Reprocess (equivalent to Lambda's batch layer)
print(f"\n   🔄 REPROCESSING (replay all events from offset 0):")
reprocessed = kappa.reprocess()
print(f"   Reprocessed {reprocessed} events")
print(f"   orders: {kappa.query('orders')}")
print(f"   (Same result — proves correctness!)")

print("\n💡 Key Insight:")
print("   Kappa achieves 'batch accuracy' by REPLAYING the event log.")
print("   No separate batch system needed — just reset the offset and replay.")


---
# 📗 Section 4: Medallion Architecture (Most Popular)

## Overview

The Medallion Architecture organizes data into three quality layers:

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│     BRONZE      │    │     SILVER      │    │      GOLD       │
│   (Raw Data)    │───▶│ (Cleaned Data)  │───▶│ (Business Data) │
│                 │    │                 │    │                 │
│ • Exact copy    │    │ • Deduplicated  │    │ • Aggregated    │
│ • All formats   │    │ • Type-cast     │    │ • Joined        │
│ • Append-only   │    │ • Validated     │    │ • Business logic│
│ • Schema-on-    │    │ • Standardized  │    │ • Ready for BI  │
│   read          │    │ • Filtered      │    │ • Ready for ML  │
│                 │    │                 │    │                 │
│ Retention: Long │    │ Retention: Med  │    │ Retention: Short│
│ (years)         │    │ (months)        │    │ (weeks/months)  │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

## Why Medallion is So Popular

1. **Simple to understand** — three layers, clear purpose for each
2. **Works for batch AND streaming** — same pattern, different triggers
3. **Incremental by design** — each layer processes only new data
4. **Debuggable** — if Gold is wrong, check Silver; if Silver is wrong, check Bronze
5. **Reprocessable** — Bronze is immutable, so you can always rebuild Silver/Gold

## Medallion Best Practices

| Layer | Storage | Processing | Quality Checks |
|-------|---------|-----------|----------------|
| **Bronze** | Append-only, raw format | Minimal (just land it) | Schema validation only |
| **Silver** | Delta, typed, deduplicated | Clean, validate, standardize | Not-null, unique, range |
| **Gold** | Delta, aggregated, optimized | Business logic, joins, metrics | Business rules, reconciliation |

## Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| Skipping Bronze | Can't reprocess if Silver logic changes | Always keep raw data |
| Too much logic in Bronze | Bronze should be simple landing | Move logic to Silver |
| One Gold table for everything | Becomes a bottleneck | Multiple Gold tables per use case |
| No quality checks between layers | Bad data propagates | Add checks at each boundary |
| Overwriting Bronze | Lose audit trail | Always append, never overwrite |

## Production Example: Databricks Customers

Most Databricks customers use Medallion:
- **Bronze**: Auto Loader ingests files from S3/ADLS → Delta tables
- **Silver**: DLT (Declarative Pipelines) cleans and validates
- **Gold**: dbt or notebooks build business metrics
- **Serving**: SQL Warehouses serve dashboards

In [0]:
# Medallion Architecture implementation patterns
class MedallionPipeline:
    """Demonstrates Medallion Architecture patterns."""
    
    def __init__(self, domain):
        self.domain = domain
        self.bronze = []
        self.silver = []
        self.gold = {}
    
    def ingest_to_bronze(self, raw_records):
        """Bronze: Land raw data as-is (append-only)."""
        for record in raw_records:
            # Add metadata
            record["_ingested_at"] = "2024-03-15T10:00:00"
            record["_source_file"] = "s3://raw/orders/2024-03-15/part-001.json"
            record["_batch_id"] = "batch_001"
        self.bronze.extend(raw_records)
        return len(raw_records)
    
    def process_to_silver(self):
        """Silver: Clean, validate, deduplicate."""
        seen_ids = set()
        rejected = []
        
        for record in self.bronze:
            # Skip already processed
            record_id = record.get("id")
            if record_id in seen_ids:
                rejected.append({"record": record, "reason": "duplicate"})
                continue
            
            # Validate
            if record.get("amount") is None:
                rejected.append({"record": record, "reason": "null_amount"})
                continue
            
            try:
                amount = float(record["amount"])
                if amount < 0:
                    rejected.append({"record": record, "reason": "negative_amount"})
                    continue
            except (ValueError, TypeError):
                rejected.append({"record": record, "reason": "invalid_amount"})
                continue
            
            # Clean and standardize
            silver_record = {
                "id": record_id,
                "customer": record.get("customer", "").strip().title(),
                "amount": amount,
                "category": record.get("category", "unknown").lower(),
                "date": record.get("date"),
            }
            self.silver.append(silver_record)
            seen_ids.add(record_id)
        
        return len(self.silver), len(rejected)
    
    def build_gold(self):
        """Gold: Aggregate for business use."""
        # Daily revenue by category
        daily_category = {}
        for record in self.silver:
            key = (record["date"], record["category"])
            if key not in daily_category:
                daily_category[key] = {"date": record["date"], "category": record["category"],
                                       "orders": 0, "revenue": 0}
            daily_category[key]["orders"] += 1
            daily_category[key]["revenue"] += record["amount"]
        
        self.gold["daily_category_revenue"] = list(daily_category.values())
        return len(self.gold["daily_category_revenue"])


# Demo
pipeline = MedallionPipeline("orders")

# Raw data (messy, duplicates, bad records)
raw_data = [
    {"id": 1, "customer": "  alice smith  ", "amount": "100.50", "category": "ELECTRONICS", "date": "2024-03-15"},
    {"id": 2, "customer": "bob jones", "amount": "200", "category": "Clothing", "date": "2024-03-15"},
    {"id": 1, "customer": "alice smith", "amount": "100.50", "category": "electronics", "date": "2024-03-15"},  # DUPLICATE
    {"id": 3, "customer": "charlie", "amount": None, "category": "food", "date": "2024-03-15"},  # NULL amount
    {"id": 4, "customer": "diana", "amount": "-50", "category": "returns", "date": "2024-03-15"},  # NEGATIVE
    {"id": 5, "customer": "eve", "amount": "300", "category": "Electronics", "date": "2024-03-15"},
]

print("🥉🥈🥇 MEDALLION ARCHITECTURE DEMO")
print("=" * 60)

# Bronze
bronze_count = pipeline.ingest_to_bronze(raw_data)
print(f"\n🥉 BRONZE: Ingested {bronze_count} raw records (as-is, with metadata)")

# Silver
silver_count, rejected_count = pipeline.process_to_silver()
print(f"\n🥈 SILVER: {silver_count} clean records ({rejected_count} rejected)")
print(f"   Rejections: duplicate, null_amount, negative_amount")
for r in pipeline.silver:
    print(f"   {r}")

# Gold
gold_count = pipeline.build_gold()
print(f"\n🥇 GOLD: {gold_count} aggregated records")
for r in pipeline.gold["daily_category_revenue"]:
    print(f"   {r['date']} | {r['category']:<12} | orders={r['orders']} | revenue=${r['revenue']:.2f}")


---
# 📗 Section 5: Data Mesh

## Overview

Data Mesh is a **decentralized** approach where each business domain owns its data end-to-end.

### The Four Principles of Data Mesh

| Principle | What It Means | Example |
|-----------|--------------|---------|
| **Domain Ownership** | Each team owns their data pipelines | Orders team owns order data |
| **Data as a Product** | Treat data like a product (SLAs, docs, quality) | "Orders API" with 99.9% uptime |
| **Self-Serve Platform** | Shared infrastructure, self-service tools | Central Databricks platform |
| **Federated Governance** | Global standards, local implementation | Naming conventions, PII rules |

### Data Mesh vs Centralized

```
CENTRALIZED (traditional):              DATA MESH (decentralized):
┌─────────────────────────┐            ┌──────┐ ┌──────┐ ┌──────┐
│   Central Data Team     │            │Orders│ │Users │ │Payments│
│                         │            │ Team │ │ Team │ │  Team  │
│ Owns ALL pipelines      │            │      │ │      │ │        │
│ Bottleneck for requests │            │Own   │ │Own   │ │Own     │
│ Doesn't understand      │            │data  │ │data  │ │data    │
│ domain context          │            │end-  │ │end-  │ │end-    │
│                         │            │to-end│ │to-end│ │to-end  │
└─────────────────────────┘            └──┬───┘ └──┬───┘ └──┬─────┘
                                          │        │        │
                                    ┌─────▼────────▼────────▼─────┐
                                    │   Self-Serve Data Platform   │
                                    │   (shared infrastructure)    │
                                    └─────────────────────────────┘
```

### When to Use Data Mesh

| Factor | Use Data Mesh | Don't Use Data Mesh |
|--------|--------------|-------------------|
| **Org size** | 500+ engineers, 10+ domains | < 100 engineers |
| **Data maturity** | High (existing pipelines, governance) | Low (still building basics) |
| **Domain complexity** | Many distinct business domains | Simple, unified domain |
| **Team autonomy** | Teams want ownership | Central team works fine |
| **Bottleneck** | Central data team is overwhelmed | No bottleneck |

### Production Example: Zalando

Zalando (European e-commerce) implemented Data Mesh:
- **Domains**: Catalog, Orders, Payments, Logistics, Marketing (15+ domains)
- **Platform**: Shared Databricks + Kafka infrastructure
- **Products**: Each domain publishes "data products" with SLAs
- **Governance**: Central standards (naming, PII handling), local implementation
- **Result**: 3x faster time-to-insight, reduced central team bottleneck by 80%

In [0]:
# Data Mesh simulation
class DataProduct:
    """A data product in a Data Mesh architecture."""
    
    def __init__(self, name, domain, owner, sla_freshness_hours, quality_threshold):
        self.name = name
        self.domain = domain
        self.owner = owner
        self.sla_freshness_hours = sla_freshness_hours
        self.quality_threshold = quality_threshold
        self.consumers = []
        self.schema = {}
    
    def add_consumer(self, consumer_domain, use_case):
        self.consumers.append({"domain": consumer_domain, "use_case": use_case})
    
    def describe(self):
        return {
            "name": self.name,
            "domain": self.domain,
            "owner": self.owner,
            "sla": f"Updated every {self.sla_freshness_hours} hours",
            "quality": f">{self.quality_threshold}% completeness",
            "consumers": len(self.consumers)
        }


# Build a Data Mesh for an e-commerce company
data_products = [
    DataProduct("orders_daily", "Orders", "orders-team@company.com", 
                sla_freshness_hours=1, quality_threshold=99.9),
    DataProduct("customer_360", "Customers", "customer-team@company.com",
                sla_freshness_hours=4, quality_threshold=99.5),
    DataProduct("product_catalog", "Catalog", "catalog-team@company.com",
                sla_freshness_hours=24, quality_threshold=99.0),
    DataProduct("payment_transactions", "Payments", "payments-team@company.com",
                sla_freshness_hours=0.5, quality_threshold=99.99),
    DataProduct("shipping_events", "Logistics", "logistics-team@company.com",
                sla_freshness_hours=2, quality_threshold=99.0),
]

# Add cross-domain consumers
data_products[0].add_consumer("Analytics", "Revenue dashboards")
data_products[0].add_consumer("ML", "Demand forecasting")
data_products[1].add_consumer("Marketing", "Personalization")
data_products[1].add_consumer("Orders", "Customer validation")
data_products[3].add_consumer("Orders", "Payment confirmation")
data_products[3].add_consumer("Finance", "Reconciliation")

print("🕸️ DATA MESH — Data Products Catalog")
print("=" * 70)
print(f"{'Product':<25} {'Domain':<12} {'SLA':<20} {'Quality':<12} {'Consumers'}")
print("-" * 70)
for dp in data_products:
    desc = dp.describe()
    print(f"{desc['name']:<25} {desc['domain']:<12} {desc['sla']:<20} "
          f"{desc['quality']:<12} {desc['consumers']}")

print("\n📊 Cross-Domain Dependencies:")
for dp in data_products:
    if dp.consumers:
        print(f"   {dp.name} → consumed by:")
        for c in dp.consumers:
            print(f"      • {c['domain']}: {c['use_case']}")

print("\n💡 Data Mesh Key Insight:")
print("   Each domain OWNS their data product end-to-end:")
print("   • They build the pipeline")
print("   • They ensure quality (SLA)")
print("   • They document it")
print("   • They support consumers")
print("   The platform team provides shared infrastructure, not pipelines.")


---
# 📗 Section 6: Architecture Decision Framework

## How to Choose the Right Architecture

### Decision Tree

```
START: What are your requirements?
│
├── Need real-time AND complete historical?
│   ├── YES → Can you maintain two codebases?
│   │   ├── YES → LAMBDA
│   │   └── NO → MEDALLION with streaming Silver
│   └── NO → Continue...
│
├── Is everything event-driven / streaming?
│   ├── YES → Do you have a replayable event log (Kafka)?
│   │   ├── YES → KAPPA
│   │   └── NO → MEDALLION with streaming
│   └── NO → Continue...
│
├── Do you have 500+ engineers and 10+ domains?
│   ├── YES → Is your central data team a bottleneck?
│   │   ├── YES → DATA MESH
│   │   └── NO → MEDALLION (centralized)
│   └── NO → Continue...
│
└── Default → MEDALLION (works for 90% of cases)
```

### Quick Reference

| Architecture | Best For | Complexity | Team Size |
|-------------|----------|-----------|-----------|
| **Medallion** | Most use cases, getting started | Low | Any |
| **Lambda** | Batch + real-time, fault tolerance | High | 10+ DEs |
| **Kappa** | Event-driven, streaming-first | Medium | 5+ DEs |
| **Data Mesh** | Large orgs, many domains | Very High | 50+ DEs |
| **Hybrid** | Complex requirements | Varies | Varies |

In [0]:
# Architecture decision engine
def recommend_architecture(requirements):
    """Recommend architecture based on requirements."""
    
    scores = {"Medallion": 5, "Lambda": 0, "Kappa": 0, "Data Mesh": 0}
    reasons = {"Medallion": ["Default choice (works for 90% of cases)"]}
    
    # Real-time requirements
    if requirements.get("real_time_latency_seconds", 999) < 5:
        scores["Kappa"] += 4
        scores["Lambda"] += 3
        reasons.setdefault("Kappa", []).append("Sub-5-second latency needed")
    
    # Historical reprocessing
    if requirements.get("needs_historical_reprocessing"):
        scores["Lambda"] += 3
        scores["Medallion"] += 2
        reasons.setdefault("Lambda", []).append("Historical reprocessing required")
    
    # Team size
    team_size = requirements.get("data_team_size", 5)
    if team_size > 50:
        scores["Data Mesh"] += 4
        reasons.setdefault("Data Mesh", []).append(f"Large team ({team_size} DEs)")
    elif team_size < 5:
        scores["Medallion"] += 3
        scores["Lambda"] -= 2
        scores["Data Mesh"] -= 3
    
    # Number of domains
    if requirements.get("num_domains", 1) > 10:
        scores["Data Mesh"] += 3
        reasons.setdefault("Data Mesh", []).append(f"{requirements['num_domains']} domains")
    
    # Event-driven
    if requirements.get("event_driven"):
        scores["Kappa"] += 3
        reasons.setdefault("Kappa", []).append("Event-driven architecture")
    
    # Simplicity priority
    if requirements.get("simplicity_priority"):
        scores["Medallion"] += 3
        scores["Lambda"] -= 2
        scores["Data Mesh"] -= 2
    
    # Sort and recommend
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    winner = ranked[0][0]
    
    print(f"   Recommendation: {winner}")
    print(f"   Scores: {dict(ranked)}")
    if winner in reasons:
        print(f"   Reasons: {'; '.join(reasons[winner])}")
    
    return winner


# Test scenarios
scenarios = [
    {
        "name": "Small startup, 3 engineers, daily reports",
        "reqs": {"data_team_size": 3, "simplicity_priority": True, 
                 "real_time_latency_seconds": 3600}
    },
    {
        "name": "Ride-sharing company, real-time pricing",
        "reqs": {"data_team_size": 30, "real_time_latency_seconds": 1,
                 "event_driven": True, "needs_historical_reprocessing": True}
    },
    {
        "name": "Large bank, 200 DEs, 15 business domains",
        "reqs": {"data_team_size": 200, "num_domains": 15,
                 "needs_historical_reprocessing": True}
    },
    {
        "name": "IoT platform, all streaming, event-driven",
        "reqs": {"data_team_size": 15, "real_time_latency_seconds": 2,
                 "event_driven": True}
    },
]

print("🏗️ ARCHITECTURE DECISION ENGINE")
print("=" * 60)
for scenario in scenarios:
    print(f"\n📌 {scenario['name']}")
    recommend_architecture(scenario["reqs"])


---
# 📗 Section 7: Practice Exercise

## Design an Architecture

**Scenario:** You're the lead data engineer at a mid-size fintech company (150 employees, 10 DEs).

**Requirements:**
- Process 50M payment transactions/day
- Real-time fraud detection (< 500ms latency)
- Daily regulatory reports (must be 100% accurate)
- ML team needs features for credit scoring (updated hourly)
- 3 business domains: Payments, Risk, Compliance
- Budget: $80K/month for data platform

**Design the architecture:**

In [0]:
# ============================================
# 🎯 YOUR TURN: Design a Fintech Architecture
# ============================================
# Consider:
# 1. Which architecture pattern(s)? (can be hybrid!)
# 2. What technology for each component?
# 3. How do you handle the real-time fraud requirement?
# 4. How do you ensure 100% accuracy for regulatory reports?
# 5. How do you serve ML features?
#
# Write your architecture design below:


In [0]:
# ============================================
# ✅ SOLUTION: Fintech Architecture Design
# ============================================
fintech_architecture = {
    "pattern": "HYBRID: Medallion + Kappa (for fraud)",
    "rationale": "Medallion for batch/reporting, Kappa stream for real-time fraud",
    
    "components": {
        "Real-time Fraud (Kappa)": {
            "ingestion": "Kafka (payment events, < 100ms)",
            "processing": "Flink or Spark Structured Streaming",
            "serving": "Redis (feature cache) + Fraud API",
            "latency": "< 500ms end-to-end",
            "accuracy": "95%+ (speed over perfection)"
        },
        "Batch Pipeline (Medallion)": {
            "bronze": "Auto Loader → Delta (raw transactions)",
            "silver": "DLT pipeline (clean, validate, deduplicate)",
            "gold": "dbt models (regulatory reports, analytics)",
            "latency": "Hourly refresh",
            "accuracy": "100% (reconciled with source)"
        },
        "ML Features": {
            "source": "Silver layer (clean transactions)",
            "processing": "Hourly Spark job → Feature Store",
            "serving": "Databricks Feature Store → ML models",
            "latency": "1 hour freshness"
        },
        "Regulatory Reporting": {
            "source": "Gold layer (reconciled data)",
            "processing": "Daily batch (must complete by 6 AM)",
            "validation": "Row-count reconciliation + business rules",
            "serving": "SQL Warehouse → Compliance dashboards"
        }
    },
    
    "technology_stack": {
        "Streaming": "Kafka + Spark Structured Streaming",
        "Batch": "Databricks (Spark + Delta Lake)",
        "Orchestration": "Lakeflow Jobs (Databricks Workflows)",
        "Storage": "Delta Lake on S3",
        "Governance": "Unity Catalog",
        "ML": "Databricks Feature Store + MLflow",
        "BI": "SQL Warehouse + Tableau"
    },
    
    "cost_estimate": {
        "Streaming (always-on)": "$25K/month",
        "Batch compute": "$20K/month",
        "Storage": "$10K/month",
        "SQL Warehouse": "$15K/month",
        "ML compute": "$10K/month",
        "Total": "$80K/month (within budget)"
    }
}

print("🏦 FINTECH ARCHITECTURE DESIGN")
print("=" * 70)
print(f"\n   Pattern: {fintech_architecture['pattern']}")
print(f"   Rationale: {fintech_architecture['rationale']}")

print("\n📌 COMPONENTS:")
for name, details in fintech_architecture["components"].items():
    print(f"\n   {name}:")
    for key, value in details.items():
        print(f"      {key}: {value}")

print("\n📌 COST ESTIMATE:")
for item, cost in fintech_architecture["cost_estimate"].items():
    print(f"   {item:<30} {cost}")


---
# 📗 Section 8: Architecture Decision Summary

## Quick Reference: Which Architecture to Use

```
START HERE: What's your primary requirement?

Need real-time AND complete historical accuracy?
  YES -> Lambda Architecture
  NO  -> Continue...

Is everything event-driven / streaming?
  YES -> Do you have a replayable event log (Kafka)?
    YES -> Kappa Architecture
    NO  -> Medallion with streaming
  NO  -> Continue...

Large org (500+ engineers) with many domains?
  YES -> Is central data team a bottleneck?
    YES -> Data Mesh
    NO  -> Medallion (centralized)
  NO  -> Continue...

DEFAULT -> Medallion Architecture (works for 90% of cases)
```

## Architecture Comparison Matrix

| Aspect | Medallion | Lambda | Kappa | Data Mesh |
|--------|-----------|--------|-------|-----------|
| Complexity | Low | High | Medium | Very High |
| Latency | Minutes-Hours | Seconds+Hours | Seconds | Varies |
| Team Size | Any | 10+ DEs | 5+ DEs | 50+ DEs |
| Reprocessing | Easy | Easy (batch) | Replay stream | Per domain |
| Cost | Low-Medium | High | Medium | High |
| Best for | Most cases | Batch+RT | Event-driven | Large orgs |

## Hybrid Patterns

Most production systems combine architectures:

```
Example: E-commerce platform
  - Medallion for batch analytics (orders, revenue)
  - Kappa for real-time fraud detection
  - Data Mesh for domain ownership (orders, payments, catalog)
```

## The Lakehouse Advantage

The lakehouse eliminates the warehouse vs lake debate:

| Feature | Data Lake | Data Warehouse | Lakehouse |
|---------|-----------|---------------|-----------|
| Storage cost | Low | High | Low |
| ACID transactions | No | Yes | Yes |
| Schema enforcement | No | Yes | Yes |
| ML support | Yes | Limited | Yes |
| BI performance | Poor | Excellent | Excellent |
| Open formats | Yes | No | Yes |

In [0]:
# Architecture decision tool
def recommend_architecture(requirements):
    scores = {"Medallion": 5, "Lambda": 0, "Kappa": 0, "Data Mesh": 0}
    
    if requirements.get("real_time_latency_seconds", 999) < 5:
        scores["Kappa"] += 4
        scores["Lambda"] += 3
    
    if requirements.get("needs_historical_reprocessing"):
        scores["Lambda"] += 3
        scores["Medallion"] += 2
    
    team_size = requirements.get("data_team_size", 5)
    if team_size > 50:
        scores["Data Mesh"] += 4
    elif team_size < 5:
        scores["Medallion"] += 3
        scores["Data Mesh"] -= 3
    
    if requirements.get("num_domains", 1) > 10:
        scores["Data Mesh"] += 3
    
    if requirements.get("event_driven"):
        scores["Kappa"] += 3
    
    if requirements.get("simplicity_priority"):
        scores["Medallion"] += 3
        scores["Lambda"] -= 2
        scores["Data Mesh"] -= 2
    
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[0][0], dict(ranked)

# Test scenarios
scenarios = [
    ("Small startup, 3 engineers", {"data_team_size": 3, "simplicity_priority": True}),
    ("Ride-sharing, real-time pricing", {"data_team_size": 30, "real_time_latency_seconds": 1, "event_driven": True}),
    ("Large bank, 200 DEs, 15 domains", {"data_team_size": 200, "num_domains": 15}),
    ("IoT platform, all streaming", {"data_team_size": 15, "real_time_latency_seconds": 2, "event_driven": True}),
]

print("ARCHITECTURE RECOMMENDATIONS")
print("=" * 60)
for name, reqs in scenarios:
    winner, scores = recommend_architecture(reqs)
    print(f"\n{name}:")
    print(f"  Recommended: {winner}")
    print(f"  Scores: {scores}")


---
*Notebook 47 of the Databricks Data Engineering series*
*Study Order: 47 (Expert Topics)*

---
# 📗 Summary

## Architecture Comparison

| Architecture | Complexity | Latency | Accuracy | Best For |
|-------------|-----------|---------|----------|----------|
| **Medallion** | Low | Minutes-Hours | High | 90% of use cases |
| **Lambda** | High | Seconds + Hours | Highest | Batch + Real-time |
| **Kappa** | Medium | Seconds | High | Event-driven |
| **Data Mesh** | Very High | Varies | Varies | Large organizations |
| **Hybrid** | Varies | Varies | Varies | Complex requirements |

## Key Takeaways

1. **Start with Medallion** — it works for most cases and is simplest
2. **Add streaming when needed** — don't over-engineer from day 1
3. **Lambda for compliance** — when you need both speed AND accuracy
4. **Kappa for event-driven** — when everything is a stream
5. **Data Mesh for scale** — only when centralized team is a bottleneck
6. **Hybrid is normal** — most production systems combine patterns

---
*Notebook 49 of the Databricks Data Engineering series*
*Study Order: 47 (Advanced Topics)*